# 0. Overview

Moisture-flux analysis using ERA5 vertically integrated water-vapour flux.

# 1. Import Library


In [ ]:
from pathlib import Path

# Define data paths (relative to notebook location)
RAINFALL_PATH = Path('../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('../external/ClimateData/era5-monthly/era5monthly_uvq_1980-2020.nc').resolve()
MFC_PATH = Path('../external/ClimateData/era5-monthly/vimf.nc').resolve()
SVP_PATH = Path('analysis_26-14_rev2/psi_chi_windparts_850.nc').resolve()
NINO34_PATH = Path('../external/ClimateData/index-monthly/nino34.anom.csv').resolve()

import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'


# 2. Open Data & Pre-Process

#### Open ERA5 Moisture Flux

In [ ]:
era5_path = mfc_path = MFC_PATH
era5_chunks = {'valid_time': 12}

ds_era5 = xr.open_dataset(era5_path, chunks=era5_chunks)[['viwve', 'viwvn']]
ds_era5 = ds_era5.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_era5 = ds_era5.assign_coords(lon=(ds_era5.lon % 360)).sortby('lon')
# slice wilayah maritime continent samudra pasifik dan samudra hindia
ds_era5 = ds_era5.sel(
    time=slice('1980-12-01', '2020-02-01'),
    lat=slice(21, -21),
    lon=slice(70, 290),
)
# keep DJF only, with December assigned to the following DJF year
month_mask = ds_era5.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(ds_era5.time.dt.month == 12, ds_era5.time.dt.year + 1, ds_era5.time.dt.year)
ds_era5 = ds_era5.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

viwve = ds_era5['viwve']
viwvn = ds_era5['viwvn']

a = 6.371e6

# xarray-friendly MFC helper on a regular lat-lon grid
# differentiate() returns per-degree gradients, so convert to radians explicitly.
def calc_mfc(qx, qy):
    phi = np.deg2rad(qx['lat'])
    dqx_dlam = qx.differentiate('lon') * (180.0 / np.pi)
    dqycos_dphi = (qy * np.cos(phi)).differentiate('lat') * (180.0 / np.pi)
    div_q = (1.0 / (a * np.cos(phi))) * dqx_dlam + (1.0 / (a * np.cos(phi))) * dqycos_dphi
    mfc = (-div_q).rename('mfc')
    mfc.attrs['units'] = 'kg m^-2 s^-1'
    mfc.attrs['long_name'] = 'moisture flux convergence'
    return mfc


In [ ]:
ds_era5 = xr.open_dataset(era5_path, chunks=era5_chunks)[['viwve', 'viwvn']]

ds_era5

#### Open NINO34 index

In [ ]:
nino34_path = nino34_path = NINO34_PATH
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2020-02-29']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)


#### Define Period

In [ ]:
full_years = np.arange(1981, 2021)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2021)

mfc_clim = calc_mfc(viwve, viwvn)

mfc_full = mfc_clim.sel(time=mfc_clim.djf_year.isin(full_years))
mfc_past = mfc_full.sel(time=mfc_full.djf_year.isin(past_years))
mfc_recent = mfc_full.sel(time=mfc_full.djf_year.isin(recent_years))

qx_clim = viwve.sel(time=viwve.djf_year.isin(full_years))
qx_past = qx_clim.sel(time=qx_clim.djf_year.isin(past_years))
qx_recent = qx_clim.sel(time=qx_clim.djf_year.isin(recent_years))

qy_clim = viwvn.sel(time=viwvn.djf_year.isin(full_years))
qy_past = qy_clim.sel(time=qy_clim.djf_year.isin(past_years))
qy_recent = qy_clim.sel(time=qy_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()


#### Define Area

In [ ]:
# Define the extent and ticks 
# ip = Indonesia-Pacific region
ip_extent = [70, 290, -21, 21]
ip_yticks = [-15, 0, 15]
ip_xticks = np.arange(90, 291, 30)


In [ ]:
# Define the extent and ticks 
# mc: maritime continent
mc_extent = [90, 155, -15, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-10, 11, 10)
mc_xticks = np.arange(100, 151, 10)


# 3. Plot Climatology

In [ ]:
# Compute the shared climatology means once and reuse the period split
full_plot, past_plot, recent_plot, full_qx_plot, past_qx_plot, recent_qx_plot, full_qy_plot, past_qy_plot, recent_qy_plot, full_mfc_plot, past_mfc_plot, recent_mfc_plot = dask.compute(
    mfc_clim.mean('time'),
    mfc_past.mean('time'),
    mfc_recent.mean('time'),
    qx_clim.mean('time'),
    qx_past.mean('time'),
    qx_recent.mean('time'),
    qy_clim.mean('time'),
    qy_past.mean('time'),
    qy_recent.mean('time'),
    mfc_clim.mean('time'),
    mfc_past.mean('time'),
    mfc_recent.mean('time'),
)

# Transpose after loading (faster on in-memory numpy arrays)
full_plot = full_plot.transpose('lat', 'lon')
past_plot = past_plot.transpose('lat', 'lon')
recent_plot = recent_plot.transpose('lat', 'lon')
diff_plot = recent_plot - past_plot

full_qx_plot = full_qx_plot.transpose('lat', 'lon')
past_qx_plot = past_qx_plot.transpose('lat', 'lon')
recent_qx_plot = recent_qx_plot.transpose('lat', 'lon')
diff_qx_plot = recent_qx_plot - past_qx_plot

full_qy_plot = full_qy_plot.transpose('lat', 'lon')
past_qy_plot = past_qy_plot.transpose('lat', 'lon')
recent_qy_plot = recent_qy_plot.transpose('lat', 'lon')
diff_qy_plot = recent_qy_plot - past_qy_plot

# keep the dedicated MFC fields explicit for plotting and later reuse
full_mfc_plot = full_mfc_plot.transpose('lat', 'lon')
past_mfc_plot = past_mfc_plot.transpose('lat', 'lon')
recent_mfc_plot = recent_mfc_plot.transpose('lat', 'lon')
diff_mfc_plot = recent_mfc_plot - past_mfc_plot


In [ ]:
def _add_quiver(ax, u, v, step=16, scale=70, width=0.002, color='k', key_u=10, key_y=1.1, key_label=None, key_color='k'):
    u_plot = u.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    v_plot = v.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=width,
        color=color,
        zorder=4,
    )
    if key_label is None:
        key_label = f'{key_u} kg m$^{{-1}}$ s$^{{-1}}$'
    ax.quiverkey(q, X=0.88, Y=key_y, U=key_u, label=key_label, labelpos='E', coordinates='axes', color=key_color)
    return q


def add_quiver_clim(ax, u, v, step=16, scale=70, width=0.002, color='k', key_y=1.1, key_u=10, key_label=None):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, key_u=key_u, key_y=key_y, key_label=key_label)


def add_quiver_anom(ax, u, v, step=16, scale=70, width=0.002, key_u=2, color='gray', key_color='black', key_label=None, key_y=1.1):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, 
                       key_u=key_u, key_y=key_y, key_label=key_label, key_color=key_color)


### Indo Pasifik

In [ ]:
mfc_absmax = 1e-4 * 1e3
mfc_levels = np.linspace(-mfc_absmax, mfc_absmax, 21)
mfc_ticks = np.linspace(-mfc_absmax, mfc_absmax, 9)
plots = [
    (full_plot * 1e3, full_qx_plot, full_qy_plot, '(a) DJF 1981-2020', 'RdBu_r', mfc_levels, mfc_ticks, 'MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', 'both'),
    (past_plot * 1e3, past_qx_plot, past_qy_plot, '(b) DJF 1981-2006', 'RdBu_r', mfc_levels, mfc_ticks, 'MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', 'both'),
    (recent_plot * 1e3, recent_qx_plot, recent_qy_plot, '(c) DJF 2007-2020', 'RdBu_r', mfc_levels, mfc_ticks, 'MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', 'both'),
    (diff_plot * 1e3, diff_qx_plot, diff_qy_plot, '(d) Selisih (c)-(b)', 'RdBu_r', mfc_levels, mfc_ticks, 'Selisih MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', 'both'),
]

for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )
    # step: distance between arrows larger means fewer arrows,
    # scale: adjust arrow length larger means shorter arrows,
    # width: arrow thickness
    if title.startswith('(d)'):
        add_quiver_anom(ax, u_data, v_data, step=25, scale=2000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=25, scale=5000, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$')

    ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


## Maritime Continent

In [ ]:
for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    # step: distance between arrows larger means fewer arrows,
    # scale: adjust arrow length larger means shorter arrows,
    # width: arrow thickness
    if data is diff_plot:
        add_quiver_anom(ax, u_data, v_data, step=5, scale=9000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black', key_y=1.02)
    else:
        add_quiver_clim(ax, u_data, v_data, step=10, scale=4500, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$', key_y=1.02)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 4. Plot Composite

### A. Analisis El Nino

In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2020):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2020):', elnino_years_recent)

mfc_clim_elnino = mfc_clim.sel(time=mfc_clim.djf_year.isin(elnino_years_clim))
mfc_past_elnino = mfc_past.sel(time=mfc_past.djf_year.isin(elnino_years_past))
mfc_recent_elnino = mfc_recent.sel(time=mfc_recent.djf_year.isin(elnino_years_recent))

qx_clim_elnino = qx_clim.sel(time=qx_clim.djf_year.isin(elnino_years_clim))
qx_past_elnino = qx_past.sel(time=qx_past.djf_year.isin(elnino_years_past))
qx_recent_elnino = qx_recent.sel(time=qx_recent.djf_year.isin(elnino_years_recent))

qy_clim_elnino = qy_clim.sel(time=qy_clim.djf_year.isin(elnino_years_clim))
qy_past_elnino = qy_past.sel(time=qy_past.djf_year.isin(elnino_years_past))
qy_recent_elnino = qy_recent.sel(time=qy_recent.djf_year.isin(elnino_years_recent))

mfc_clim_elnino, mfc_past_elnino, mfc_recent_elnino, qx_clim_elnino, qx_past_elnino, qx_recent_elnino, qy_clim_elnino, qy_past_elnino, qy_recent_elnino = dask.compute(
    mfc_clim_elnino.mean('time'),
    mfc_past_elnino.mean('time'),
    mfc_recent_elnino.mean('time'),
    qx_clim_elnino.mean('time'),
    qx_past_elnino.mean('time'),
    qx_recent_elnino.mean('time'),
    qy_clim_elnino.mean('time'),
    qy_past_elnino.mean('time'),
    qy_recent_elnino.mean('time'),
)

mfc_clim_elnino = mfc_clim_elnino.transpose('lat', 'lon')
mfc_past_elnino = mfc_past_elnino.transpose('lat', 'lon')
mfc_recent_elnino = mfc_recent_elnino.transpose('lat', 'lon')

qx_clim_elnino = qx_clim_elnino.transpose('lat', 'lon')
qx_past_elnino = qx_past_elnino.transpose('lat', 'lon')
qx_recent_elnino = qx_recent_elnino.transpose('lat', 'lon')

qy_clim_elnino = qy_clim_elnino.transpose('lat', 'lon')
qy_past_elnino = qy_past_elnino.transpose('lat', 'lon')
qy_recent_elnino = qy_recent_elnino.transpose('lat', 'lon')

mfc_clim_elnino_anom = mfc_clim_elnino - full_plot
mfc_past_elnino_anom = mfc_past_elnino - past_plot
mfc_recent_elnino_anom = mfc_recent_elnino - recent_plot

diff_elnino_anom = mfc_recent_elnino_anom - mfc_past_elnino_anom

mfc_clim_elnino_anom = mfc_clim_elnino_anom.reset_coords(drop=True)
mfc_past_elnino_anom = mfc_past_elnino_anom.reset_coords(drop=True)
mfc_recent_elnino_anom = mfc_recent_elnino_anom.reset_coords(drop=True)


#### plot komposit indo pasifik

In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (mfc_clim_elnino_anom * 1e3, qx_clim_elnino, qy_clim_elnino, '(a) Komposit El Nino DJF (1981 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_past_elnino_anom * 1e3, qx_past_elnino, qy_past_elnino, '(b) Komposit El Nino DJF (1981 - 2006)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_recent_elnino_anom * 1e3, qx_recent_elnino, qy_recent_elnino, '(c) Komposit El Nino DJF (2007 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (diff_elnino_anom * 1e3, (qx_recent_elnino - qx_past_elnino), (qy_recent_elnino - qy_past_elnino), '(d) Selisih (c) - (b)', 'RdBu_r', 'Selisih MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
]

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )
    
    if title.startswith('(d)'):
        add_quiver_anom(ax, u_data, v_data, step=25, scale=2000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=25, scale=5000, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot mc

In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (mfc_clim_elnino_anom * 1e3, qx_clim_elnino, qy_clim_elnino, '(a) Komposit El Nino DJF (1981 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_past_elnino_anom * 1e3, qx_past_elnino, qy_past_elnino, '(b) Komposit El Nino DJF (1981 - 2006)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_recent_elnino_anom * 1e3, qx_recent_elnino, qy_recent_elnino, '(c) Komposit El Nino DJF (2007 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (diff_elnino_anom * 1e3, (qx_recent_elnino - qx_past_elnino), (qy_recent_elnino - qy_past_elnino), '(d) Selisih (c) - (b)', 'RdBu_r', 'Selisih MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
]

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )
    
    if title.startswith('(d)'):
        add_quiver_anom(ax, u_data, v_data, step=5, scale=3000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black', key_y=1.02)
    else:
        add_quiver_clim(ax, u_data, v_data, step=10, scale=4500, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$', key_y=1.02)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if data is diff_elnino_anom:
        add_quiver_anom(ax, u_data, v_data, step=5, scale=9000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black', key_y=1.02)
    else:
        add_quiver_clim(ax, u_data, v_data, step=10, scale=4500, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$', key_y=1.02)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


### B. Analisis La Nina

In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf[nino34_column] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf[nino34_column] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf[nino34_column] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2020):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2020):', lanina_years_recent)

mfc_clim_lanina = mfc_clim.sel(time=mfc_clim.djf_year.isin(lanina_years_clim))
mfc_past_lanina = mfc_past.sel(time=mfc_past.djf_year.isin(lanina_years_past))
mfc_recent_lanina = mfc_recent.sel(time=mfc_recent.djf_year.isin(lanina_years_recent))

qx_clim_lanina = qx_clim.sel(time=qx_clim.djf_year.isin(lanina_years_clim))
qx_past_lanina = qx_past.sel(time=qx_past.djf_year.isin(lanina_years_past))
qx_recent_lanina = qx_recent.sel(time=qx_recent.djf_year.isin(lanina_years_recent))

qy_clim_lanina = qy_clim.sel(time=qy_clim.djf_year.isin(lanina_years_clim))
qy_past_lanina = qy_past.sel(time=qy_past.djf_year.isin(lanina_years_past))
qy_recent_lanina = qy_recent.sel(time=qy_recent.djf_year.isin(lanina_years_recent))

mfc_clim_lanina, mfc_past_lanina, mfc_recent_lanina, qx_clim_lanina, qx_past_lanina, qx_recent_lanina, qy_clim_lanina, qy_past_lanina, qy_recent_lanina = dask.compute(
    mfc_clim_lanina.mean('time'),
    mfc_past_lanina.mean('time'),
    mfc_recent_lanina.mean('time'),
    qx_clim_lanina.mean('time'),
    qx_past_lanina.mean('time'),
    qx_recent_lanina.mean('time'),
    qy_clim_lanina.mean('time'),
    qy_past_lanina.mean('time'),
    qy_recent_lanina.mean('time'),
)

mfc_clim_lanina = mfc_clim_lanina.transpose('lat', 'lon')
mfc_past_lanina = mfc_past_lanina.transpose('lat', 'lon')
mfc_recent_lanina = mfc_recent_lanina.transpose('lat', 'lon')

qx_clim_lanina = qx_clim_lanina.transpose('lat', 'lon')
qx_past_lanina = qx_past_lanina.transpose('lat', 'lon')
qx_recent_lanina = qx_recent_lanina.transpose('lat', 'lon')

qy_clim_lanina = qy_clim_lanina.transpose('lat', 'lon')
qy_past_lanina = qy_past_lanina.transpose('lat', 'lon')
qy_recent_lanina = qy_recent_lanina.transpose('lat', 'lon')

mfc_clim_lanina_anom = mfc_clim_lanina - full_plot
mfc_past_lanina_anom = mfc_past_lanina - past_plot
mfc_recent_lanina_anom = mfc_recent_lanina - recent_plot
diff_lanina_anom = mfc_recent_lanina_anom - mfc_past_lanina_anom

mfc_clim_lanina_anom = mfc_clim_lanina_anom.reset_coords(drop=True)
mfc_past_lanina_anom = mfc_past_lanina_anom.reset_coords(drop=True)
mfc_recent_lanina_anom = mfc_recent_lanina_anom.reset_coords(drop=True)


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (mfc_clim_lanina_anom * 1e3, qx_clim_lanina, qy_clim_lanina, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_past_lanina_anom * 1e3, qx_past_lanina, qy_past_lanina, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (mfc_recent_lanina_anom * 1e3, qx_recent_lanina, qy_recent_lanina, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu_r', 'Anomali MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
    (diff_lanina_anom * 1e3, (qx_recent_lanina - qx_past_lanina), (qy_recent_lanina - qy_past_lanina), '(d) Selisih (c) - (b)', 'RdBu_r', 'Selisih MFC (10$^{-3}$ kg m$^{-2}$ s$^{-1}$)', np.arange(-0.1, 0.1001, 0.01), np.arange(-0.1, 0.1001, 0.05), 'both'),
]


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if title.startswith('(d)'):
        add_quiver_anom(ax, u_data, v_data, step=25, scale=2000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=25, scale=5000, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if title.startswith('(d)'):
        add_quiver_anom(ax, u_data, v_data, step=5, scale=3000, width=0.0017, color='black',
                        key_u=50, key_label='50 kg m$^{-1}$ s$^{-1}$', key_color='black', key_y=1.02)
    else:
        add_quiver_clim(ax, u_data, v_data, step=10, scale=4500, width=0.0017,
                        key_u=100, key_label='100 kg m$^{-1}$ s$^{-1}$', key_y=1.02)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 5. Plot Correlation

Compute the DJF teleconnection as Pearson correlation between DJF-mean MFC and DJF-mean Niño3.4, then plot the correlation maps.

In [ ]:
# Compute DJF-mean MFC and DJF-mean Niño3.4, then correlate by djf_year
mfc_clim_djf_corr = mfc_clim.groupby('djf_year').mean('time')
mfc_past_djf_corr = mfc_past.groupby('djf_year').mean('time')
mfc_recent_djf_corr = mfc_recent.groupby('djf_year').mean('time')

nino34_clim_djf_corr_series = nino34_clim.groupby('djf_year')[nino34_column].mean()
nino34_past_djf_corr_series = nino34_past.groupby('djf_year')[nino34_column].mean()
nino34_recent_djf_corr_series = nino34_recent.groupby('djf_year')[nino34_column].mean()

nino34_clim_djf_corr = xr.DataArray(
    nino34_clim_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_clim_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_past_djf_corr = xr.DataArray(
    nino34_past_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_past_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_recent_djf_corr = xr.DataArray(
    nino34_recent_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_recent_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)

mfc_clim_djf_corr, nino34_clim_djf_corr = xr.align(mfc_clim_djf_corr, nino34_clim_djf_corr, join='inner')
mfc_past_djf_corr, nino34_past_djf_corr = xr.align(mfc_past_djf_corr, nino34_past_djf_corr, join='inner')
mfc_recent_djf_corr, nino34_recent_djf_corr = xr.align(mfc_recent_djf_corr, nino34_recent_djf_corr, join='inner')

corr_clim, corr_past, corr_recent = dask.compute(
    xr.corr(mfc_clim_djf_corr, nino34_clim_djf_corr, dim='djf_year'),
    xr.corr(mfc_past_djf_corr, nino34_past_djf_corr, dim='djf_year'),
    xr.corr(mfc_recent_djf_corr, nino34_recent_djf_corr, dim='djf_year'),
)
corr_recent_minus_past = corr_recent - corr_past

print('Sample size full:', int(mfc_clim_djf_corr.sizes['djf_year']))
print('Sample size past:', int(mfc_past_djf_corr.sizes['djf_year']))
print('Sample size recent:', int(mfc_recent_djf_corr.sizes['djf_year']))


In [ ]:
# Student t-test significance for the DJF teleconnection maps
from scipy.stats import t as student_t, norm

def corr_pvalue(r, n):
    r = xr.where(np.abs(r) >= 0.999999, np.sign(r) * 0.999999, r)
    t_stat = r * np.sqrt((n - 2) / (1 - r ** 2))
    return xr.apply_ufunc(
        lambda x: 2 * student_t.sf(np.abs(x), df=n - 2),
        t_stat,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    )

# Fisher's z-test for difference in correlation coefficients between two independent samples
def corr_diff_pvalue(r1, n1, r2, n2):
    r1 = xr.where(np.abs(r1) >= 0.999999, np.sign(r1) * 0.999999, r1)
    r2 = xr.where(np.abs(r2) >= 0.999999, np.sign(r2) * 0.999999, r2)
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    z_stat = (z2 - z1) / np.sqrt(1 / (n2 - 3) + 1 / (n1 - 3))
    return xr.apply_ufunc(
        lambda x: 2 * norm.sf(np.abs(x)),
        z_stat,
        vectorize=True,
        dask='allowed',
        output_dtypes=[float],
    )

n_full = int(mfc_clim_djf_corr.sizes["djf_year"])
n_past = int(mfc_past_djf_corr.sizes["djf_year"])
n_recent = int(mfc_recent_djf_corr.sizes["djf_year"])

corr_clim_p = corr_pvalue(corr_clim, n_full)
corr_past_p = corr_pvalue(corr_past, n_past)
corr_recent_p = corr_pvalue(corr_recent, n_recent)
corr_recent_minus_past_p = corr_diff_pvalue(corr_past, n_past, corr_recent, n_recent)

corr_clim_sig = corr_clim_p < 0.05
corr_past_sig = corr_past_p < 0.05
corr_recent_sig = corr_recent_p < 0.05
corr_recent_minus_past_sig = corr_recent_minus_past_p < 0.05


In [ ]:
def add_stippling(ax, sig_mask, block=8, size=15, alpha=0.9):
    sig = sig_mask.fillna(False).astype(np.int8)

    # build a much coarser plotting mask
    sig_plot = (sig.coarsen(lat=block, lon=block, boundary='trim').max() > 0)

    yy, xx = np.where(sig_plot.values)
    if yy.size == 0:
        return
    ax.scatter(
        sig_plot['lon'].values[xx],
        sig_plot['lat'].values[yy],
        s=size,
        c='k',
        marker='.',
        linewidths=0,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )

for name, sig in [('full', corr_clim_sig), ('past', corr_past_sig), ('recent', corr_recent_sig), ('diff', corr_recent_minus_past_sig)]:
    frac = float(sig.fillna(False).astype(int).mean().compute()) * 100
    print(f'{name}: {frac:.1f}% significant at native grid')


In [ ]:
#plot indo pasifik

corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
diff_levels = np.arange(-1, 1.01, 0.05)
diff_ticks = np.arange(-1, 1.01, 0.25)

plot_defs = [
    (corr_clim, corr_clim_sig, '(a) Korelasi MFC vs Niño3.4 DJF 1981-2020', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_past, corr_past_sig, '(b) Korelasi MFC vs Niño3.4 DJF 1981-2006', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent, corr_recent_sig, '(c) Korelasi MFC vs Niño3.4 DJF 2007-2020', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent_minus_past, corr_recent_minus_past_sig, '(d) Selisih (c) - (b)', 'PiYG', 'Selisih korelasi', diff_levels, diff_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_stippling(ax, sig_mask, block=15, size=15, alpha=0.7)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
# plot maritime continent

corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
diff_levels = np.arange(-1, 1.01, 0.05)
diff_ticks = np.arange(-1, 1.01, 0.25)

plot_defs = [
    (corr_clim, corr_clim_sig, '(a) Korelasi MFC vs Niño3.4 DJF 1981-2020', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_past, corr_past_sig, '(b) Korelasi MFC vs Niño3.4 DJF 1981-2006', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent, corr_recent_sig, '(c) Korelasi MFC vs Niño3.4 DJF 2007-2020', 'PiYG', 'Koefisien korelasi', corr_levels, corr_ticks, 'both'),
    (corr_recent_minus_past, corr_recent_minus_past_sig, '(d) Selisih (c) - (b)', 'PiYG', 'Selisih korelasi', diff_levels, diff_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_stippling(ax, sig_mask, block=7, size=20, alpha=0.7)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 6. Plot Regresi

In [ ]:
# Reuse the DJF-mean MFC and DJF-mean Niño3.4 from the correlation section, then regress MFC on Niño3.4 by djf_year
from scipy.stats import linregress

def _linregress_1d(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    result = linregress(x[mask], y[mask])
    return result.slope, result.intercept, result.rvalue, result.pvalue, result.stderr

mfc_clim_djf_reg, nino34_clim_djf_reg = xr.align(mfc_clim_djf_corr, nino34_clim_djf_corr, join='inner')
mfc_past_djf_reg, nino34_past_djf_reg = xr.align(mfc_past_djf_corr, nino34_past_djf_corr, join='inner')
mfc_recent_djf_reg, nino34_recent_djf_reg = xr.align(mfc_recent_djf_corr, nino34_recent_djf_corr, join='inner')

mfc_clim_djf_reg = mfc_clim_djf_reg.chunk({'djf_year': -1})
nino34_clim_djf_reg = nino34_clim_djf_reg.chunk({'djf_year': -1})
mfc_past_djf_reg = mfc_past_djf_reg.chunk({'djf_year': -1})
nino34_past_djf_reg = nino34_past_djf_reg.chunk({'djf_year': -1})
mfc_recent_djf_reg = mfc_recent_djf_reg.chunk({'djf_year': -1})
nino34_recent_djf_reg = nino34_recent_djf_reg.chunk({'djf_year': -1})

reg_clim = xr.apply_ufunc(
    _linregress_1d,
    nino34_clim_djf_reg,
    mfc_clim_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_past = xr.apply_ufunc(
    _linregress_1d,
    nino34_past_djf_reg,
    mfc_past_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_recent = xr.apply_ufunc(
    _linregress_1d,
    nino34_recent_djf_reg,
    mfc_recent_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)

reg_clim_slope, reg_clim_intercept, reg_clim_r, reg_clim_p, reg_clim_stderr = dask.compute(*reg_clim)
reg_past_slope, reg_past_intercept, reg_past_r, reg_past_p, reg_past_stderr = dask.compute(*reg_past)
reg_recent_slope, reg_recent_intercept, reg_recent_r, reg_recent_p, reg_recent_stderr = dask.compute(*reg_recent)
reg_recent_minus_past_slope = reg_recent_slope - reg_past_slope

reg_clim_sig = reg_clim_p < 0.05
reg_past_sig = reg_past_p < 0.05
reg_recent_sig = reg_recent_p < 0.05

reg_absmax = xr.concat([
    np.abs(reg_clim_slope),
    np.abs(reg_past_slope),
    np.abs(reg_recent_slope),
    np.abs(reg_recent_minus_past_slope),
], dim='stack').max()
reg_absmax = float(reg_absmax)
reg_absmax = max(reg_absmax, 5e-5)
reg_absmax = np.ceil(reg_absmax / 5e-5) * 5e-5
reg_levels = np.linspace(-reg_absmax, reg_absmax, 21)
reg_ticks = np.linspace(-reg_absmax, reg_absmax, 9)

print('Sample size full:', int(mfc_clim_djf_reg.sizes['djf_year']))
print('Sample size past:', int(mfc_past_djf_reg.sizes['djf_year']))
print('Sample size recent:', int(mfc_recent_djf_reg.sizes['djf_year']))


In [ ]:
reg_levels = np.arange(-4e-4, 4.01e-4, 2e-5)
reg_ticks = np.arange(-4e-4, 4.01e-4, 1e-4)


In [ ]:
# plot regression indo pacific

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi MFC vs Niño3.4 DJF 1981-2020', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi MFC vs Niño3.4 DJF 1981-2006', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi MFC vs Niño3.4 DJF 2007-2020', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'PiYG_r', 'Selisih kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=20, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
# plot regression maritime continent

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi MFC vs Niño3.4 DJF 1981-2020', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi MFC vs Niño3.4 DJF 1981-2006', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi MFC vs Niño3.4 DJF 2007-2020', 'PiYG', 'Kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'PiYG_r', 'Selisih kemiringan regresi MFC (kg m$^{-2}$ s$^{-1}$ per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=7, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()
